## inspect_run
Read-only verifier for the `error_testing` harness. Shows the `pipeline_log` row and `pipeline_step_log` rows for a run. `pipeline_run_id_filter` defaults to `latest` (most recent `pipeline_log` row). A few seconds of serverless; writes nothing.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
dbutils.widgets.text("pipeline_run_id_filter", "latest")
PR = dbutils.widgets.get("pipeline_run_id_filter").strip()
if PR == "" or PR.lower() == "latest":
    # Most recently started run.
    PR = spark.sql(
        f"SELECT pipeline_run_id FROM {AUDIT}.pipeline_log ORDER BY started_timestamp DESC LIMIT 1"
    ).collect()[0][0]
print(f"inspect_run: pipeline_run_id={PR}")

In [ ]:
# Run-level verdict + captured error.
display(spark.sql(f"""
    SELECT pipeline_run_id, status, started_timestamp, ended_timestamp, error_message
    FROM {AUDIT}.pipeline_log
    WHERE pipeline_run_id = '{PR}'
"""))

In [ ]:
# Per-step rows (only StepLog tasks write here; pre-logging tasks intentionally do not).
display(spark.sql(f"""
    SELECT step_sequence, notebook_name, status, rows_read, rows_written, error_message
    FROM {AUDIT}.pipeline_step_log
    WHERE pipeline_run_id = '{PR}'
    ORDER BY step_sequence
"""))